# 판본이 바뀌는 문서에서 근거 찾기

문서가 더 늘었는데 오답도 늘었다. 구판과 개정판이 함께 있는 자료에서 질문 날짜에 맞는 문단과 원문 위치를 복원한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 검색 순위가 높아도 쓸 수 없는 문서

6월 안내와 7월 안내에 모두 “열람 기한”이라는 말이 들어 있다. 두 문서가 질문과 비슷하다는 사실만으로 적용할 판본을 결정할 수 없다.

원문을 읽을 때 문서 ID, 쪽, 대상, 유효 기간을 문단과 함께 보관한다. PDF 두 파일의 1쪽은 서로 다른 출처이고, 쪽 번호 하나만 저장하면 인용을 검산할 수 없다.

자른 조각으로 답을 만들기 전에 대상과 날짜를 확인한다. 후보가 사라졌다면 검색기 설정을 무조건 넓히기 전에 실제로 적용할 문서가 있는지부터 본다.

## 경계일에는 어떤 조항을 인용할까요?

유효 기간은 시작일을 포함하고 종료일을 제외한다. 점수가 높은 순서로 정렬된 후보에 기간 조건을 적용한다.

In [1]:
from datetime import date
docs = [{"id":"old:p1","start":"2026-06-01","end":"2026-07-01","days":30},
        {"id":"new:p4","start":"2026-07-01","end":"2026-10-01","days":90}]
def eligible(as_of):
    day = date.fromisoformat(as_of)
    return [d for d in docs if date.fromisoformat(d["start"]) <= day < date.fromisoformat(d["end"])]

print([(d["id"], d["days"]) for d in eligible("2026-07-01")])

[('new:p4', 90)]


구판의 종료일과 개정판의 시작일이 겹쳐 보이지만 이날 유효한 것은 개정판뿐이다. 실제 원문의 “까지”가 종료일 포함을 뜻하면 자료를 적재할 때부터 구간 표현을 맞춘다.

### 첫 판본 이전의 질문 보류하기

5월 문의에는 적용할 문서가 없다. 검색 후보가 없을 때만 보류하고 답과 출처를 함께 반환한다.

In [2]:
def answer(as_of):
    # eligible의 결과를 확인하세요.
    pass

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>def answer(as_of):
    matches = eligible(as_of)
    if not matches:
        return {&quot;status&quot;:&quot;보류&quot;, &quot;citation&quot;:None}
    doc = matches[0]
    return {&quot;status&quot;:&quot;안내&quot;, &quot;days&quot;:doc[&quot;days&quot;], &quot;citation&quot;:doc[&quot;id&quot;]}
print(answer(&quot;2026-05-20&quot;))</code></pre><pre>{&#x27;status&#x27;: &#x27;보류&#x27;, &#x27;citation&#x27;: None}
</pre><p>근거가 없는데 구판이라도 골라 주면 보류할 질문이 임의의 기한 안내로 바뀐다.</p></details>

## 겹치는 청크를 원문에 다시 꽂기

문자열이 여러 번 나오면 find로 위치를 다시 추정하는 방법이 틀릴 수 있다. 나눌 때 보관한 반열린 구간을 원문에 적용한다.

In [3]:
text = "공지: 열람 기간은 30일. 개정: 열람 기간은 90일."
spans = [(0,16),(12,len(text))]
chunks = [{"id":f"p4-{i}", "start":a, "end":b, "text":text[a:b]}
          for i,(a,b) in enumerate(spans)]
hit = chunks[1]
print(hit["id"], repr(text[hit["start"]:hit["end"]]))

p4-1 '0일. 개정: 열람 기간은 90일.'


문장 중간이 잘린 조각도 어디에서 왔는지는 보존된다. 검색에 잡힌 짧은 조각만 인용하면 첫 번째 판본의 30일과 개정판의 90일이 섞일 수 있다.

### 같은 쪽의 겹침을 중복 없이 복원하기

두 청크를 문자열로 더하지 말고 같은 원문의 오프셋으로 연속 구간을 복원한다. 겹침이 사라졌을 때 전체 문장과 같은지 확인한다.

In [4]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>a,b = spans[0]
c,d = spans[1]
if a &lt;= c &lt;= b:
    restored = text[a:max(b,d)]
    print(restored, restored == text)</code></pre><pre>공지: 열람 기간은 30일. 개정: 열람 기간은 90일. True
</pre><p>같은 원문에서 연속일 때만 이런 병합이 가능하다. 다른 파일이나 다른 쪽의 오프셋은 먼저 분리해야 한다.</p></details>

## 다른 입력에 적용하기

운영 편람 PDF의 같은 용어가 나오는 행사 안내와 업무 규정을 함께 검색한다. 문서 종류를 바꾸기 전후의 후보 ID를 적고, 남은 청크에서 답 문장이 잘렸다면 인접 문단까지 원문에서 복원한다.

질문만 바꾼 실험과 청크 크기만 바꾼 실험을 분리한다. 인용 링크를 따라가서 실제 쪽에 같은 문장이 있는지 사람이 확인할 수 있어야 한다.